### 批量提取并导出有用曲线

（已废弃）

- 输入目录：`data/vertical_well_las_vp`
- 输出目录：`data/vertical_well_las_vp_vs_rho_interpre_raw`
- 规则来源：`import.csv`
- 提取类别：井眼质量、伽马/泥质、纵波声波、横波声波、密度、孔隙度、渗透率、含水饱和度
- 策略：每井按 `import.csv` 中该 8 类的全部 mnemonic 尝试提取，缺失曲线静默跳过并记录。


In [ ]:
import sys
from pathlib import Path

import lasio
import pandas as pd

# 兼容从工作区根目录或 notebooks 目录启动内核。
project_root = Path.cwd().resolve()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from cup.petrel.export import export_logsets_to_las
from cup.petrel.load import extract_any_log_from_las

input_dir = project_root / "data" / "all_well_las_vp"
output_dir = project_root / "data" / "vertical_well_las_vp_vs_rho_interpre_raw"
import_csv_path = project_root / "data" / "import.csv"

selected_categories = [
    "井眼质量",
    "伽马/泥质",
    "纵波声波",
    "横波声波",
    "密度",
    "孔隙度",
    "渗透率",
    "含水饱和度",
]


def parse_mnemonics(raw_value: object) -> list[str]:
    if raw_value is None or pd.isna(raw_value):  # type: ignore
        return []
    text = str(raw_value).strip()
    if not text or text == "-":
        return []

    out: list[str] = []
    for token in text.split(","):
        mnemonic = token.strip()
        if mnemonic and mnemonic != "-":
            out.append(mnemonic)
    return out


import_df = pd.read_csv(import_csv_path)
if "井名" not in import_df.columns:
    raise ValueError("import.csv 缺少 '井名' 列。")

missing_cols = [c for c in selected_categories if c not in import_df.columns]
if missing_cols:
    raise ValueError(f"import.csv 缺少类别列: {missing_cols}")

logsets_for_export: dict[str, dict] = {}
skipped_wells_local: list[dict[str, str]] = []
skipped_curves_local: list[dict[str, str]] = []

for _, row in import_df.iterrows():
    well_name = str(row["井名"]).strip()
    if not well_name or well_name == "nan":
        continue

    las_path = input_dir / f"{well_name}.las"
    if not las_path.exists():
        skipped_wells_local.append({"well": well_name, "reason": f"未找到 LAS 文件: {las_path.name}"})
        continue

    las_file = lasio.read(las_path)
    extracted_logs = {}

    for category in selected_categories:
        for mnemonic in parse_mnemonics(row[category]):
            if mnemonic in extracted_logs:
                continue
            try:
                extracted_logs[mnemonic] = extract_any_log_from_las(las_file, mnemonic)
            except Exception as exc:
                skipped_curves_local.append(
                    {
                        "well": well_name,
                        "curve": mnemonic,
                        "reason": str(exc),
                    }
                )

    if not extracted_logs:
        skipped_wells_local.append({"well": well_name, "reason": "8类目标曲线均未成功提取"})
        continue

    logsets_for_export[well_name] = extracted_logs

export_summary = export_logsets_to_las(
    logsets=logsets_for_export,  # type: ignore
    output_dir=output_dir,
    curve_names=None,
    null_value=-999.25,
)

summary = {
    "planned_wells": int(import_df["井名"].notna().sum()),
    "prepared_for_export": len(logsets_for_export),
    "exported_files": len(export_summary["exported_files"]),
    "local_skipped_wells": len(skipped_wells_local),
    "local_skipped_curves": len(skipped_curves_local),
    "export_skipped_wells": len(export_summary["skipped_wells"]),
    "export_skipped_curves": len(export_summary["skipped_curves"]),
}

pd.DataFrame([summary])